In [ ]:
import pandas as pd
import numpy as np
import math
import os
import glob
import uproot as r
import awkward as ak
import vector
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
a1MeV = pd.read_csv('./train_data/select_data/m_a = 1   MeV.csv')
a10MeV = pd.read_csv('./train_data/select_data/m_a = 10  MeV.csv')
a100MeV = pd.read_csv('./train_data/select_data/m_a = 100 MeV.csv')
a1GeV = pd.read_csv('./train_data/select_data/m_a = 1   GeV.csv')
a10GeV = pd.read_csv('./train_data/select_data/m_a = 10  GeV.csv')
aME = pd.read_csv('./train_data/select_data/Mono-gamma.csv')

In [ ]:
data_dict = {
    R'$m_{a} = 1  $ MeV': a1MeV,        # 信号类
    R'$m_{a} = 10 $ MeV': a10MeV,        # 信号类
    R'$m_{a} = 100$ MeV': a100MeV,        # 信号类
    R'$m_{a} = 0.8  $ GeV': a1GeV,        # 信号类
    R'$m_{a} = 10 $ GeV': a10GeV,        # 信号类
    R'$Mono-\gamma$': aME,      # 背景类
}

In [ ]:
def combined_plot(name, name1, Min_range, Max_range, Bin, 
                  data_dict, name2=None, n_col=2, log='log', loc_s = [0.3, 0.2], loc_pro = [0.3, 0.1]):
    """
    绘制归一化直方图，并根据信号和背景设置线条实线和虚线。

    参数:
        name (str): 保存图像的基本名称。
        name1 (str): x轴的名称。
        Min_range, Max_range (float): 横坐标范围的最小值和最大值。
        Bin (int): 直方图的分箱数。
        data_dict (dict): 数据字典，包含要绘制的数据集，键为标签，值为 DataFrame。
        name2 (str): 用于组合绘图的第二个变量名（可选）。
        n_col (int): 图例的列数。
        log (str): y轴比例，可选 'linear' 或 'log'。
    """
    plt.figure(figsize=(5, 3.1))  # 调整图像大小

    # 定义颜色
    colors = ['black', 'blue', 'red', 'green', 'magenta', 'darkorange']

    # 遍历数据集，绘制每组直方图
    for i, (label, data) in enumerate(data_dict.items()):
        # 获取当前数据列
        values = data[f"{name}"]

        # 计算频数和边界
        counts, bins = np.histogram(values, bins=Bin, range=(Min_range, Max_range))
        total_count = sum(counts)  # 总事件数

        # 判断线条样式：信号为实线，背景为虚线
        linestyle = '-' if '_' in label.lower() else '-'  # 信号为实线，背景为虚线
        linewidth = 1.5 if '_' in label.lower() else 1.5     # 信号线宽较大，背景线宽较小

        # 使用 plt.hist 绘制概率直方图轮廓
        plt.hist(values, bins=Bin, range=(Min_range, Max_range), 
                 weights=[1 / total_count for _ in values],  # 使用归一化权重
                 histtype='step', linewidth=linewidth, 
                 color=colors[i % len(colors)], linestyle=linestyle, 
                 label=f"{label}")

    # 设置图表标题和坐标轴
    plt.legend(ncol=n_col)
    plt.xlabel(f'{name1} [GeV]', fontsize=15)
    plt.yscale(f'{log}')
    # 设置刻度字体大小
    plt.tick_params(axis='x', labelsize=12)  # x 轴刻度字体大小
    plt.tick_params(axis='y', labelsize=12)  # y 轴刻度字体大小
    plt.ylabel("Normalized Events", fontsize=15)
    #plt.text(loc_s[0], loc_s[1], R'$\sqrt{S} = 14$ TeV', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
    #plt.text(loc_pro[0], loc_pro[1], 'Opposite Sign', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
    plt.tight_layout()
    plt.grid(False)  # 关闭网格

    # 保存图像
    plt.savefig(f"./plot/Normalized Histogram of {name}.png", dpi=300)
    plt.show()  # 显示图像
def combined_plot_R(name, name1, Min_range, Max_range, Bin, 
                  data_dict, name2=None, n_col=2, log='log', loc_s = [0.3, 0.2], loc_pro = [0.3, 0.1]):
    """
    绘制归一化直方图，并根据信号和背景设置线条实线和虚线。

    参数:
        name (str): 保存图像的基本名称。
        name1 (str): x轴的名称。
        Min_range, Max_range (float): 横坐标范围的最小值和最大值。
        Bin (int): 直方图的分箱数。
        data_dict (dict): 数据字典，包含要绘制的数据集，键为标签，值为 DataFrame。
        name2 (str): 用于组合绘图的第二个变量名（可选）。
        n_col (int): 图例的列数。
        log (str): y轴比例，可选 'linear' 或 'log'。
    """
    plt.figure(figsize=(5, 3.1))  # 调整图像大小

    # 定义颜色
    colors = ['black', 'blue', 'red', 'green', 'magenta', 'darkorange']

    # 遍历数据集，绘制每组直方图
    for i, (label, data) in enumerate(data_dict.items()):
        # 获取当前数据列
        values = data[f"{name}"]

        # 计算频数和边界
        counts, bins = np.histogram(values, bins=Bin, range=(Min_range, Max_range))
        total_count = sum(counts)  # 总事件数

        # 判断线条样式：信号为实线，背景为虚线
        linestyle = '-' if '_' in label.lower() else '-'  # 信号为实线，背景为虚线
        linewidth = 1.5 if '_' in label.lower() else 1.5     # 信号线宽较大，背景线宽较小
        nonzero_values = [v for v in values if v != 0]
        total_count = len(nonzero_values)
        weights = [1 / total_count for _ in nonzero_values]
        # 使用 plt.hist 绘制概率直方图轮廓
        plt.hist(values, bins=Bin, range=(Min_range, Max_range), 
                 weights = weights,
                 histtype='step', linewidth=linewidth, 
                 color=colors[i % len(colors)], linestyle=linestyle, 
                 label=f"{label}")

    # 设置图表标题和坐标轴
    plt.legend(ncol=n_col)
    plt.xlabel(f'{name1}', fontsize=15)
    plt.yscale(f'{log}')
    # 设置刻度字体大小
    plt.tick_params(axis='x', labelsize=12)  # x 轴刻度字体大小
    plt.tick_params(axis='y', labelsize=12)  # y 轴刻度字体大小
    plt.ylabel("Normalized Events", fontsize=15)
    #plt.text(loc_s[0], loc_s[1], R'$\sqrt{S} = 14$ TeV', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
    #plt.text(loc_pro[0], loc_pro[1], 'Opposite Sign', ha='center', va='center', transform=plt.gca().transAxes, fontsize=14, color='black')
    plt.tight_layout()
    plt.grid(False)  # 关闭网格
    #print(f"原始长度: {len(values)}, 非零长度: {len(nonzero_values)}")

    # 保存图像
    plt.savefig(f"./plot/Normalized Histogram of {name}.png", dpi=300)
    plt.show()  # 显示图像

In [ ]:
a100MeV.columns

In [ ]:
combined_plot(name = 'MissingET_MET_0',name1=R'$E^{T}_{miss}$',Min_range=0.1, Max_range=50, Bin=100,
              data_dict=data_dict, log='linear', n_col=1)

In [ ]:
combined_plot(name = 'photon_energy',name1=r'$E_{\gamma}$',Min_range=0.1, Max_range=55, Bin=100,
              data_dict=data_dict, log='linear',n_col=1)
combined_plot_R(name = 'photon_theta_cos',name1=r'$|cos(\theta_{\gamma})|$',Min_range=0.1, Max_range=1, Bin=40,
              data_dict=data_dict, log='linear')